# 03 — Финальная модель прогнозирования

## Цель

В этом ноутбуке остаются только две наиболее содержательные forecasting-идеи проекта.

**Часть A. Cross-sectional residual correction**

Сильный baseline из трёх моделей остаётся основой прогноза. CatBoost не заменяет его, а учится делать только небольшую локальную поправку для конкретного муниципального ряда.

Контрольный результат на стандартном Jun–Sep benchmark:

- baseline ensemble: **761.916 MAE**
- наша residual correction: **758.339 MAE**

**Часть B. Online adaptive ensemble**

На более позднем forward-периоде веса пяти экспертов пересчитываются только по уже завершившимся прогнозам того же горизонта.

Контрольный результат:

- baseline ensemble: **880.397 MAE**
- static 45/55: **858.810 MAE**
- наш online adaptive ensemble: **834.497 MAE**

## Методологическое правило

Исторические прогнозы формируются с отсечением входной истории на origin.
Однако настройка residual correction включает фактические значения за июнь 2024 года,
поэтому Jun–Sep benchmark рассматривается как ретроспективный challenger,
а не как полностью независимый real-time holdout с 1 июня.

Jun–Sep уже исследовался в ходе разработки проекта, поэтому результат **758.339** корректно трактовать как challenger benchmark, а не как новый untouched holdout.

Поздний Oct–Nov forward-период используется отдельно для проверки переноса адаптивной архитектуры.

Результаты двух периодов нельзя напрямую сравнивать между собой, потому что это разные временные окна.

## 1. Импорты и входные артефакты

Ноутбук использует результаты предыдущих финальных этапов:

- `00_data_final.ipynb` — панель данных;
- `01_forecasting_baselines_final.ipynb` — прогнозы baseline-моделей;
- `02_forecasting_static_final.ipynb` — Panel AR, Seasonal и статический ансамбль.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import torch

from catboost import CatBoostRegressor
from chronos import BaseChronosPipeline
from scipy.optimize import minimize
from statsforecast import StatsForecast
from statsforecast.models import AutoETS

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 220)

# Переносимое определение корня репозитория (запуск из корня или notebooks/).
_CURRENT_DIR = Path.cwd().resolve()
if (_CURRENT_DIR / "data").is_dir() and (_CURRENT_DIR / "notebooks").is_dir():
    PROJECT_DIR = _CURRENT_DIR
elif (_CURRENT_DIR.parent / "data").is_dir() and (_CURRENT_DIR.parent / "notebooks").is_dir():
    PROJECT_DIR = _CURRENT_DIR.parent
else:
    raise FileNotFoundError(
        "Корень проекта не найден. Запускайте Jupyter из корня репозитория "
        "или его каталога notebooks/."
    )
INTERIM_DIR = PROJECT_DIR / "data" / "interim"

# Необязательный аудит общих параметров. Отсутствие YAML не препятствует запуску:
# значения моделей остаются зафиксированными в исходном эксперименте.
CONFIG_FILE = PROJECT_DIR / "configs" / "final_config.yaml"
if CONFIG_FILE.exists():
    try:
        import yaml
    except ImportError as exc:
        raise ImportError("Установите PyYAML для чтения configs/final_config.yaml") from exc
    with CONFIG_FILE.open("r", encoding="utf-8") as f:
        PROJECT_CONFIG = yaml.safe_load(f) or {}
    _horizons = (PROJECT_CONFIG.get("forecasting") or {}).get("horizons")
    if _horizons is not None and list(_horizons) != [1, 2, 3]:
        raise ValueError("Config: forecasting.horizons должен быть [1, 2, 3] для frozen benchmark")
    _chronos = (PROJECT_CONFIG.get("chronos") or {}).get("checkpoint")
    if _chronos is not None and _chronos != "amazon/chronos-bolt-base":
        raise ValueError("Config: chronos.checkpoint отличается от frozen notebook 03")
    print("Конфигурация прочитана; ключевые параметры согласованы")
else:
    PROJECT_CONFIG = {}
    print("Config YAML пока отсутствует: используются исходные frozen параметры")

WIDE_FILE = INTERIM_DIR / "consumption_total_wide_2016x24.parquet"
PANEL_FILE = INTERIM_DIR / "consumption_total_panel_2016.parquet"

BASELINE_FILE = INTERIM_DIR / "baseline_multihorizon_predictions.parquet"
STATIC_TEST_FILE = INTERIM_DIR / "our_static_multihorizon_predictions.parquet"
STATIC_FORWARD_FILE = INTERIM_DIR / "our_static_forward_nov_dec_predictions.parquet"

GEO_FILE = INTERIM_DIR / "consumption_series_geography.parquet"
CLUSTER_FILE = INTERIM_DIR / "seasonal_cluster_map_k5.parquet"

required = [
    WIDE_FILE,
    PANEL_FILE,
    BASELINE_FILE,
    STATIC_TEST_FILE,
    STATIC_FORWARD_FILE,
    GEO_FILE,
    CLUSTER_FILE,
]

missing = [
    str(path)
    for path in required
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Не найдены необходимые входные файлы:\n"
        + "\n".join(missing)
    )

print("Входные артефакты: OK")


In [ ]:
wide_export = pd.read_parquet(WIDE_FILE)

wide_ref = (
    wide_export
    .set_index("series_id")
    .copy()
)

wide_ref.index.name = None
wide_ref.columns = pd.to_datetime(wide_ref.columns)
wide_ref = wide_ref.sort_index(axis=1)

panel = pd.read_parquet(PANEL_FILE)
panel["period"] = pd.to_datetime(panel["period"])

baseline_test = pd.read_parquet(BASELINE_FILE)
static_test = pd.read_parquet(STATIC_TEST_FILE)
static_forward = pd.read_parquet(STATIC_FORWARD_FILE)

geography = pd.read_parquet(GEO_FILE)
cluster_map = pd.read_parquet(CLUSTER_FILE)

for df in [baseline_test, static_test, static_forward]:
    for col in ["origin", "target_period"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col])

print("wide_ref:", wide_ref.shape)
print("baseline_test:", baseline_test.shape)
print("static_test:", static_test.shape)
print("static_forward:", static_forward.shape)

assert wide_ref.shape == (2016, 24)
assert len(static_test) == 24_192
assert len(static_forward) == 6_048
assert panel["series_id"].nunique() == 2016

print("Загрузка: PASSED")


## 2. Пять экспертных моделей

Для финальной архитектуры используются пять разнородных прогнозов:

1. Panel AR
2. Seasonal
3. Chronos relative
4. ETS relative
5. Seasonal growth

Jun–Sep прогнозы уже рассчитаны в предыдущих ноутбуках.

Для ранней calibration-истории и позднего forward-периода недостающие прогнозы пересчитываются каузально.

In [ ]:
def panel_log_factor(wide):
    # Общий уровень панели в log-space.
    return np.log(
        wide.clip(lower=1e-6)
    ).median(axis=0)


def seasonal_growth_forecast(
    wide,
    horizons,
    growth_window=3,
):
    # Годовой сезонный якорь + средний недавний YoY.
    horizons = list(horizons)

    log_level = np.log(
        wide.clip(lower=1e-6)
    )

    log_yoy = (
        log_level
        - log_level.shift(
            12,
            axis=1,
        )
    )

    recent_growth = (
        log_yoy.iloc[
            :,
            -growth_window:
        ]
        .mean(axis=1)
    )

    result = {}

    for h in horizons:

        seasonal_anchor = (
            wide.iloc[
                :,
                -12 + h - 1
            ]
        )

        result[h] = (
            seasonal_anchor
            * np.exp(
                recent_growth
            )
        )

    return pd.DataFrame(
        result,
        index=wide.index,
    )


def factor_future(
    factor,
    horizons,
    growth_window=1,
):
    # Прогноз общего фактора в исходной шкале.
    factor_level = pd.DataFrame(
        [
            np.exp(
                factor.to_numpy()
            )
        ],
        columns=factor.index,
    )

    return seasonal_growth_forecast(
        factor_level,
        horizons=horizons,
        growth_window=growth_window,
    ).iloc[0]


def ets_relative_forecast(
    wide,
    horizons,
):
    # ETS прогнозирует отклонение МО от общего фактора.
    horizons = list(horizons)

    factor = panel_log_factor(
        wide
    )

    relative = (
        np.log(
            wide.clip(lower=1e-6)
        )
        .sub(
            factor,
            axis=1,
        )
    )

    long = (
        relative
        .stack()
        .rename("y")
        .reset_index()
    )

    long.columns = [
        "unique_id",
        "ds",
        "y",
    ]

    sf = StatsForecast(
        models=[
            AutoETS(
                season_length=1
            )
        ],
        freq="MS",
        n_jobs=1,
    )

    fc = sf.forecast(
        df=long.dropna(),
        h=max(horizons),
    )

    if "unique_id" not in fc.columns:
        fc = fc.reset_index()

    fc["h"] = (
        fc
        .groupby("unique_id")
        .cumcount()
        + 1
    )

    relative_future = (
        fc
        .pivot(
            index="unique_id",
            columns="h",
            values="AutoETS",
        )
    )

    factor_fc = factor_future(
        factor,
        horizons,
        growth_window=1,
    )

    result = {
        h:
            np.exp(
                relative_future[h]
            )
            * factor_fc[h]
        for h in horizons
    }

    return (
        pd.DataFrame(result)
        .reindex(
            wide.index
        )
    )


In [ ]:
print("Загрузка Chronos-Bolt-base ...")

chronos_pipeline = (
    BaseChronosPipeline
    .from_pretrained(
        "amazon/chronos-bolt-base",
        device_map="cuda",
        torch_dtype=torch.float32,
    )
)

print("Chronos готов.")


def chronos_relative_forecast(
    wide,
    horizons,
    batch_size=256,
):
    # Chronos прогнозирует относительную log-траекторию.
    horizons = list(horizons)

    factor = panel_log_factor(
        wide
    )

    relative = (
        np.log(
            wide.clip(lower=1e-6)
        )
        .sub(
            factor,
            axis=1,
        )
    )

    prediction_length = (
        max(horizons)
    )

    values = relative.to_numpy(
        dtype=np.float32
    )

    batches = []

    for start in range(
        0,
        len(relative),
        batch_size,
    ):

        block = values[
            start:
            start + batch_size
        ]

        contexts = [
            torch.tensor(
                row[
                    np.isfinite(row)
                ],
                dtype=torch.float32,
            )
            for row in block
        ]

        quantiles, _ = (
            chronos_pipeline
            .predict_quantiles(
                contexts,
                prediction_length=
                    prediction_length,
                quantile_levels=[
                    0.5
                ],
            )
        )

        batches.append(
            quantiles[
                ...,
                0
            ]
            .detach()
            .cpu()
            .numpy()
        )

    relative_future = (
        np.vstack(
            batches
        )
    )

    factor_fc = factor_future(
        factor,
        horizons,
        growth_window=1,
    )

    result = {}

    for h in horizons:

        result[h] = (
            np.exp(
                relative_future[
                    :,
                    h - 1
                ]
            )
            * factor_fc[h]
        )

    return pd.DataFrame(
        result,
        index=wide.index,
    )


### 2.1 Panel AR

Panel AR разделяет динамику на общий фактор панели и локальное отклонение конкретного МО.

In [ ]:
panel_ar_data = (
    panel
    .sort_values(
        [
            "series_id",
            "period",
        ]
    )
    .copy()
)

panel_ar_data["lag_12"] = (
    panel_ar_data
    .groupby(
        "series_id"
    )["value"]
    .shift(12)
)

panel_ar_data["log_yoy"] = np.log(
    panel_ar_data["value"]
    / panel_ar_data["lag_12"]
)

panel_factor = (
    panel_ar_data
    .dropna(
        subset=[
            "log_yoy"
        ]
    )
    .groupby(
        "period"
    )[
        "log_yoy"
    ]
    .median()
    .sort_index()
)

panel_ar_data["panel_factor"] = (
    panel_ar_data["period"]
    .map(
        panel_factor
    )
)

panel_ar_data["panel_residual"] = (
    panel_ar_data["log_yoy"]
    - panel_ar_data["panel_factor"]
)

panel_ar_data["residual_lag1"] = (
    panel_ar_data
    .groupby(
        "series_id"
    )[
        "panel_residual"
    ]
    .shift(1)
)


def forecast_panel_factor(
    origin,
    horizons,
    last_weight=0.65,
    trend_weight=0.35,
):
    # Рекурсивный прогноз общего фактора.
    origin = pd.Timestamp(
        origin
    )

    horizons = sorted(
        int(h)
        for h in horizons
    )

    history = (
        panel_factor.loc[
            panel_factor.index
            <= origin
        ]
        .astype(float)
        .tolist()
    )

    rows = []

    for h in range(
        1,
        max(horizons) + 1,
    ):

        y3 = np.asarray(
            history[-3:],
            dtype=float,
        )

        x3 = np.arange(
            3,
            dtype=float,
        )

        slope, intercept = np.polyfit(
            x3,
            y3,
            1,
        )

        trend_next = (
            intercept
            + slope * 3
        )

        next_factor = (
            last_weight
            * history[-1]
            + trend_weight
            * trend_next
        )

        history.append(
            next_factor
        )

        if h in horizons:

            rows.append({
                "h":
                    h,
                "factor_pred":
                    next_factor,
            })

    return pd.DataFrame(
        rows
    )


def forecast_panel_ar(
    origin,
    horizons,
):
    # Pooled AR(1) для локального residual.
    origin = pd.Timestamp(
        origin
    )

    horizons = sorted(
        int(h)
        for h in horizons
    )

    train = panel_ar_data[
        (
            panel_ar_data[
                "period"
            ]
            <= origin
        )
        &
        panel_ar_data[
            "panel_residual"
        ].notna()
        &
        panel_ar_data[
            "residual_lag1"
        ].notna()
    ]

    x = train[
        "residual_lag1"
    ].to_numpy()

    y = train[
        "panel_residual"
    ].to_numpy()

    beta = (
        np.sum(
            x * y
        )
        / np.sum(
            x ** 2
        )
    )

    origin_residual = (
        panel_ar_data[
            panel_ar_data[
                "period"
            ]
            == origin
        ][
            [
                "series_id",
                "panel_residual",
            ]
        ]
        .rename(
            columns={
                "panel_residual":
                    "origin_residual",
            }
        )
    )

    factor_fc = (
        forecast_panel_factor(
            origin,
            horizons,
        )
    )

    parts = []

    for h in horizons:

        target = (
            origin
            + pd.DateOffset(
                months=h
            )
        )

        seasonal_period = (
            target
            - pd.DateOffset(
                months=12
            )
        )

        base = (
            panel[
                panel[
                    "period"
                ]
                == seasonal_period
            ][
                [
                    "series_id",
                    "value",
                ]
            ]
            .rename(
                columns={
                    "value":
                        "seasonal_base",
                }
            )
        )

        factor_pred = float(
            factor_fc.loc[
                factor_fc[
                    "h"
                ]
                == h,
                "factor_pred",
            ].iloc[0]
        )

        part = (
            origin_residual
            .merge(
                base,
                on="series_id",
                how="left",
                validate="one_to_one",
            )
        )

        part["origin"] = (
            origin
        )

        part["h"] = h

        part[
            "target_period"
        ] = target

        part["panel_ar"] = (
            part[
                "seasonal_base"
            ]
            * np.exp(
                factor_pred
                + (
                    beta ** h
                )
                * part[
                    "origin_residual"
                ]
            )
        )

        parts.append(
            part[
                [
                    "series_id",
                    "origin",
                    "h",
                    "target_period",
                    "panel_ar",
                ]
            ]
        )

    out = pd.concat(
        parts,
        ignore_index=True,
    )

    assert len(out) == (
        2016
        * len(horizons)
    )

    assert out[
        "panel_ar"
    ].notna().all()

    return out


### 2.2 Seasonal expert

Сезонный эксперт использует только профили 2023 года.

Если для МО нет сезонного кластера, используется региональный профиль. Если регион также недоступен, используется общий профиль панели.

In [ ]:
frozen_meta = (
    geography[
        [
            "series_id",
            "region_code",
            "match_method",
        ]
    ]
    .copy()
)

frozen_meta[
    "frozen_region_code"
] = np.where(
    frozen_meta[
        "match_method"
    ].eq(
        "deterministic"
    ),
    frozen_meta[
        "region_code"
    ].astype(str),
    "UNK",
)

# LEFT JOIN обязателен:
# сохраняем все 2016 МО, даже если cluster_id отсутствует.
frozen_meta = (
    frozen_meta[
        [
            "series_id",
            "frozen_region_code",
        ]
    ]
    .merge(
        cluster_map[
            [
                "series_id",
                "cluster_id",
            ]
        ],
        on="series_id",
        how="left",
        validate="one_to_one",
    )
)

assert len(frozen_meta) == 2016

season_hist = (
    panel[
        [
            "series_id",
            "period",
            "value",
        ]
    ]
    .merge(
        frozen_meta,
        on="series_id",
        how="left",
        validate="many_to_one",
    )
    .sort_values(
        [
            "series_id",
            "period",
        ]
    )
    .copy()
)

season_hist["lag_1"] = (
    season_hist
    .groupby(
        "series_id"
    )[
        "value"
    ]
    .shift(1)
)

season_hist[
    "monthly_growth"
] = (
    season_hist["value"]
    / season_hist["lag_1"]
    - 1
)

season_hist["month"] = (
    season_hist["period"]
    .dt.month
)

hist_2023 = (
    season_hist[
        season_hist[
            "period"
        ].dt.year
        == 2023
    ]
    .copy()
)

national_profile = (
    hist_2023
    .groupby(
        "month"
    )[
        "monthly_growth"
    ]
    .median()
    .rename(
        "national_growth"
    )
    .reset_index()
)

region_profile = (
    hist_2023[
        hist_2023[
            "frozen_region_code"
        ]
        != "UNK"
    ]
    .groupby(
        [
            "frozen_region_code",
            "month",
        ]
    )[
        "monthly_growth"
    ]
    .median()
    .rename(
        "region_growth"
    )
    .reset_index()
)

cluster_profile = (
    hist_2023[
        hist_2023[
            "cluster_id"
        ]
        .notna()
    ]
    .groupby(
        [
            "cluster_id",
            "month",
        ]
    )[
        "monthly_growth"
    ]
    .median()
    .rename(
        "cluster_growth"
    )
    .reset_index()
)


def forecast_seasonal(
    origin,
    horizons,
):
    # Рекурсивный сезонный прогноз.
    origin = pd.Timestamp(
        origin
    )

    horizons = sorted(
        int(h)
        for h in horizons
    )

    state = (
        panel[
            panel[
                "period"
            ]
            == origin
        ][
            [
                "series_id",
                "value",
            ]
        ]
        .rename(
            columns={
                "value":
                    "recursive_value",
            }
        )
        .merge(
            frozen_meta,
            on="series_id",
            how="left",
            validate="one_to_one",
        )
    )

    parts = []

    for h in range(
        1,
        max(horizons) + 1,
    ):

        target = (
            origin
            + pd.DateOffset(
                months=h
            )
        )

        month = (
            target.month
        )

        national_growth = float(
            national_profile.loc[
                national_profile[
                    "month"
                ]
                == month,
                "national_growth",
            ].iloc[0]
        )

        step = state.copy()

        step["month"] = (
            month
        )

        step[
            "national_growth"
        ] = national_growth

        step = step.merge(
            region_profile,
            on=[
                "frozen_region_code",
                "month",
            ],
            how="left",
            validate="many_to_one",
        )

        step[
            "region_growth"
        ] = (
            step[
                "region_growth"
            ]
            .fillna(
                step[
                    "national_growth"
                ]
            )
        )

        step = step.merge(
            cluster_profile,
            on=[
                "cluster_id",
                "month",
            ],
            how="left",
            validate="many_to_one",
        )

        step[
            "cluster_growth"
        ] = (
            step[
                "cluster_growth"
            ]
            .fillna(
                step[
                    "region_growth"
                ]
            )
        )

        step[
            "seasonal_growth"
        ] = (
            0.5
            * step[
                "region_growth"
            ]
            + 0.5
            * step[
                "cluster_growth"
            ]
        )

        step["seasonal"] = (
            step[
                "recursive_value"
            ]
            * (
                1.0
                + step[
                    "seasonal_growth"
                ]
            )
        )

        if h in horizons:

            out = (
                step[
                    [
                        "series_id",
                        "seasonal",
                    ]
                ]
                .copy()
            )

            out["origin"] = (
                origin
            )

            out["h"] = h

            out[
                "target_period"
            ] = target

            parts.append(
                out
            )

        state = (
            step[
                [
                    "series_id",
                    "frozen_region_code",
                    "cluster_id",
                    "seasonal",
                ]
            ]
            .rename(
                columns={
                    "seasonal":
                        "recursive_value",
                }
            )
        )

    result = pd.concat(
        parts,
        ignore_index=True,
    )

    assert len(result) == (
        2016
        * len(horizons)
    )

    assert result[
        "seasonal"
    ].notna().all()

    return result


## 3. Общая история пяти экспертов

Нам нужны три временные зоны:

- ранняя calibration-история до Jun–Sep benchmark;
- Jun–Sep benchmark;
- поздний Oct–Nov forward-период.

После объединения одна строка соответствует одному `series_id x origin x horizon`.

In [ ]:
EXPERTS = [
    "panel_ar",
    "seasonal",
    "chronos_relative",
    "ets_relative",
    "seasonal_growth",
]


def baseline_extra_forecast(
    origin,
    horizons,
):
    # Три baseline-эксперта для дополнительных origins.
    origin = pd.Timestamp(
        origin
    )

    horizons = tuple(
        sorted(
            int(h)
            for h in horizons
        )
    )

    history = wide_ref.loc[
        :,
        wide_ref.columns
        <= origin,
    ].copy()

    chronos_fc = (
        chronos_relative_forecast(
            history,
            horizons,
        )
    )

    ets_fc = (
        ets_relative_forecast(
            history,
            horizons,
        )
    )

    snaive_fc = (
        seasonal_growth_forecast(
            history,
            horizons,
            growth_window=3,
        )
    )

    rows = []

    for h in horizons:

        target = (
            origin
            + pd.DateOffset(
                months=h
            )
        )

        rows.append(
            pd.DataFrame({
                "series_id":
                    wide_ref.index.to_numpy(),
                "origin":
                    origin,
                "h":
                    h,
                "target_period":
                    target,
                "chronos_relative":
                    chronos_fc[
                        h
                    ].to_numpy(),
                "ets_relative":
                    ets_fc[
                        h
                    ].to_numpy(),
                "seasonal_growth":
                    snaive_fc[
                        h
                    ].to_numpy(),
                "actual_value":
                    wide_ref[
                        target
                    ].to_numpy(),
            })
        )

    return pd.concat(
        rows,
        ignore_index=True,
    )


def complete_expert_forecast(
    origin,
    horizons,
):
    # Объединяем все пять экспертов.
    extra = (
        baseline_extra_forecast(
            origin,
            horizons,
        )
    )

    panel_fc = (
        forecast_panel_ar(
            origin,
            horizons,
        )
    )

    seasonal_fc = (
        forecast_seasonal(
            origin,
            horizons,
        )
    )

    return (
        extra
        .merge(
            panel_fc,
            on=[
                "series_id",
                "origin",
                "h",
                "target_period",
            ],
            how="left",
            validate="one_to_one",
        )
        .merge(
            seasonal_fc,
            on=[
                "series_id",
                "origin",
                "h",
                "target_period",
            ],
            how="left",
            validate="one_to_one",
        )
    )


In [ ]:
# Ранняя история.
# Берём только прогнозы, target_period которых не позднее 2024-06-01. Это ретроспективная календарная граница,
# а не подтверждение доступности итогов июня на начало месяца.
early_specs = [
    (
        pd.Timestamp(
            "2024-03-01"
        ),
        (1, 2, 3),
    ),
    (
        pd.Timestamp(
            "2024-04-01"
        ),
        (1, 2),
    ),
    (
        pd.Timestamp(
            "2024-05-01"
        ),
        (1,),
    ),
]

early_parts = []

for origin, horizons in early_specs:

    print(
        "Calibration origin:",
        origin.date(),
        "h=",
        horizons,
    )

    early_parts.append(
        complete_expert_forecast(
            origin,
            horizons,
        )
    )

early_history = pd.concat(
    early_parts,
    ignore_index=True,
)

print(
    "Calibration rows:",
    len(
        early_history
    ),
)

assert len(
    early_history
) == 12_096


In [ ]:
# Jun–Sep: baseline-прогнозы из notebook 01.
base_test_wide = (
    baseline_test[
        baseline_test[
            "model"
        ]
        .isin(
            [
                "chronos_relative",
                "ets_relative",
                "seasonal_growth",
            ]
        )
    ]
    .pivot(
        index=[
            "series_id",
            "origin",
            "h",
            "target_period",
            "actual_value",
        ],
        columns="model",
        values="prediction",
    )
    .reset_index()
)

base_test_wide.columns.name = None

# Panel AR и Seasonal из notebook 02.
static_test_part = (
    static_test[
        [
            "series_id",
            "origin",
            "horizon",
            "target_period",
            "panel_ar_pred",
            "seasonal_pred",
        ]
    ]
    .rename(
        columns={
            "horizon":
                "h",
            "panel_ar_pred":
                "panel_ar",
            "seasonal_pred":
                "seasonal",
        }
    )
)

mid_history = (
    base_test_wide
    .merge(
        static_test_part,
        on=[
            "series_id",
            "origin",
            "h",
            "target_period",
        ],
        how="left",
        validate="one_to_one",
    )
)

assert len(
    mid_history
) == 24_192

print(
    "Jun–Sep rows:",
    len(
        mid_history
    ),
)


In [ ]:
# Поздний forward-период.
forward_specs = [
    (
        pd.Timestamp(
            "2024-10-01"
        ),
        (1, 2),
    ),
    (
        pd.Timestamp(
            "2024-11-01"
        ),
        (1,),
    ),
]

forward_extra_parts = []

for origin, horizons in forward_specs:

    print(
        "Forward origin:",
        origin.date(),
        "h=",
        horizons,
    )

    forward_extra_parts.append(
        baseline_extra_forecast(
            origin,
            horizons,
        )
    )

forward_extra = pd.concat(
    forward_extra_parts,
    ignore_index=True,
)

static_forward_part = (
    static_forward[
        [
            "series_id",
            "origin",
            "horizon",
            "target_period",
            "panel_ar_pred",
            "seasonal_pred",
        ]
    ]
    .rename(
        columns={
            "horizon":
                "h",
            "panel_ar_pred":
                "panel_ar",
            "seasonal_pred":
                "seasonal",
        }
    )
)

forward_history = (
    forward_extra
    .merge(
        static_forward_part,
        on=[
            "series_id",
            "origin",
            "h",
            "target_period",
        ],
        how="left",
        validate="one_to_one",
    )
)

assert len(
    forward_history
) == 6_048

print(
    "Forward rows:",
    len(
        forward_history
    ),
)


In [ ]:
expert_history = pd.concat(
    [
        early_history,
        mid_history,
        forward_history,
    ],
    ignore_index=True,
)

expert_history = (
    expert_history[
        [
            "series_id",
            "origin",
            "h",
            "target_period",
            "actual_value",
            *EXPERTS,
        ]
    ]
    .sort_values(
        [
            "origin",
            "h",
            "series_id",
        ]
    )
    .reset_index(
        drop=True
    )
)

print(
    "Всего строк:",
    len(
        expert_history
    ),
)

print(
    "Forecast cells:",
    expert_history[
        [
            "origin",
            "h",
            "target_period",
        ]
    ]
    .drop_duplicates()
    .shape[0],
)

assert len(
    expert_history
) == 42_336

assert not expert_history.duplicated(
    [
        "series_id",
        "origin",
        "h",
    ]
).any()

assert expert_history[
    EXPERTS
].notna().all().all()

print(
    "История экспертов: PASSED"
)


# Часть A. Cross-sectional residual correction

## 4. Идея метода

Сильный baseline строится как среднее:

`baseline = mean(Chronos relative, ETS relative, Seasonal growth)`

CatBoost не прогнозирует расход напрямую.

Сначала на calibration-периоде считается:

`raw_residual = log(actual / baseline)`

Затем удаляется общий сдвиг внутри одной `origin x horizon` ячейки:

`cs_residual = raw_residual - median(raw_residual)`

CatBoost обучается предсказывать только этот локальный cross-sectional residual.

На inference прогноз CatBoost снова центрируется внутри `origin x horizon`, а затем применяется небольшая поправка:

`corrected_pred = baseline * exp(alpha_h * predicted_centered_residual)`

Так модель не может произвольно сдвинуть всю панель вверх или вниз.

In [ ]:
CAL_CUTOFF = pd.Timestamp(
    "2024-06-01"
)

BENCH_ORIGINS = pd.to_datetime([
    "2024-06-01",
    "2024-07-01",
    "2024-08-01",
    "2024-09-01",
])

BASELINE_EXPERTS = [
    "chronos_relative",
    "ets_relative",
    "seasonal_growth",
]

ALL_EXPERTS = EXPERTS.copy()

calibration = (
    expert_history[
        expert_history[
            "target_period"
        ]
        <= CAL_CUTOFF
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

benchmark = (
    expert_history[
        expert_history[
            "origin"
        ]
        .isin(
            BENCH_ORIGINS
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


def add_baseline(df):
    out = df.copy()

    out["baseline_pred"] = (
        out[
            BASELINE_EXPERTS
        ]
        .mean(axis=1)
    )

    out["baseline_error"] = (
        out["actual_value"]
        - out["baseline_pred"]
    ).abs()

    return out


calibration = add_baseline(
    calibration
)

benchmark = add_baseline(
    benchmark
)

print(
    "Calibration:",
    len(
        calibration
    ),
)

print(
    "Benchmark:",
    len(
        benchmark
    ),
)

assert len(
    calibration
) == 12_096

assert len(
    benchmark
) == 24_192


## 5. Causal признаки состояния ряда

Используются только признаки, известные на момент origin:

- текущий log-уровень;
- MoM;
- YoY;
- тренд последних трёх месяцев;
- волатильность;
- недавний средний YoY;
- состояние всей панели;
- размер МО по 2023 году;
- расхождение пяти экспертных прогнозов;
- percentile-положение МО внутри текущей cross-section.

`series_id` в модель не передаётся.

In [ ]:
cols_2023 = [
    c
    for c in wide_ref.columns
    if c.year == 2023
]

size_mean_2023 = (
    wide_ref[
        cols_2023
    ]
    .mean(axis=1)
)

size_group = pd.qcut(
    size_mean_2023.rank(
        method="first"
    ),
    q=5,
    labels=[
        "Q1",
        "Q2",
        "Q3",
        "Q4",
        "Q5",
    ],
)

size_map = pd.DataFrame({
    "series_id":
        wide_ref.index.to_numpy(),
    "size_mean_2023":
        size_mean_2023.to_numpy(),
    "size_group":
        size_group.astype(str).to_numpy(),
})

size_map.index.name = None

display(
    size_map[
        "size_group"
    ]
    .value_counts()
    .sort_index()
    .to_frame("n")
)

assert len(
    size_map
) == 2016


In [ ]:
def robust_mad(x):
    # Устойчивый масштаб панели.
    x = np.asarray(
        x,
        dtype=float,
    )

    med = np.nanmedian(
        x
    )

    mad = np.nanmedian(
        np.abs(
            x - med
        )
    )

    return (
        1.4826
        * mad
    )


def build_origin_state(
    origin,
):
    # Используется только история period <= origin.
    origin = pd.Timestamp(
        origin
    )

    hist = wide_ref.loc[
        :,
        wide_ref.columns
        <= origin,
    ].copy()

    log_hist = np.log(
        hist.clip(
            lower=1e-6
        )
    )

    last_log = (
        log_hist.iloc[
            :,
            -1
        ]
    )

    prev_log = (
        log_hist.iloc[
            :,
            -2
        ]
    )

    lag12 = (
        origin
        - pd.DateOffset(
            months=12
        )
    )

    if lag12 not in (
        log_hist.columns
    ):

        raise ValueError(
            f"Нет годового лага для {origin.date()}"
        )

    mom = (
        last_log
        - prev_log
    )

    yoy = (
        last_log
        - log_hist[
            lag12
        ]
    )

    trend3 = (
        log_hist.iloc[
            :,
            -1
        ]
        - log_hist.iloc[
            :,
            -3
        ]
    ) / 2.0

    diff = (
        log_hist
        .diff(
            axis=1
        )
    )

    vol3 = (
        diff.iloc[
            :,
            -3:
        ]
        .std(
            axis=1
        )
    )

    vol6 = (
        diff.iloc[
            :,
            -6:
        ]
        .std(
            axis=1
        )
    )

    yoy_hist = (
        log_hist
        - log_hist.shift(
            12,
            axis=1,
        )
    )

    yoy_mean3 = (
        yoy_hist.iloc[
            :,
            -3:
        ]
        .mean(
            axis=1
        )
    )

    out = pd.DataFrame({
        "series_id":
            wide_ref.index.to_numpy(),
        "origin":
            origin,
        "level_log":
            last_log.to_numpy(),
        "mom_log":
            mom.to_numpy(),
        "yoy_log":
            yoy.to_numpy(),
        "trend3":
            trend3.to_numpy(),
        "vol3":
            vol3.to_numpy(),
        "vol6":
            vol6.to_numpy(),
        "yoy_mean3":
            yoy_mean3.to_numpy(),
    })

    out[
        "panel_mom_median"
    ] = float(
        np.nanmedian(
            mom
        )
    )

    out[
        "panel_yoy_median"
    ] = float(
        np.nanmedian(
            yoy
        )
    )

    out[
        "panel_mom_mad"
    ] = float(
        robust_mad(
            mom
        )
    )

    out[
        "panel_yoy_mad"
    ] = float(
        robust_mad(
            yoy
        )
    )

    return out


all_origins = sorted(
    pd.concat([
        calibration[
            "origin"
        ],
        benchmark[
            "origin"
        ],
    ])
    .drop_duplicates()
    .tolist()
)

origin_state = pd.concat(
    [
        build_origin_state(
            origin
        )
        for origin
        in all_origins
    ],
    ignore_index=True,
)

origin_state = (
    origin_state
    .merge(
        size_map,
        on="series_id",
        how="left",
        validate="many_to_one",
    )
)

assert origin_state[
    "size_group"
].notna().all()

print(
    "Origin-state rows:",
    len(
        origin_state
    ),
)


In [ ]:
def add_residual_features(
    df,
):
    out = (
        df
        .merge(
            origin_state,
            on=[
                "series_id",
                "origin",
            ],
            how="left",
            validate="many_to_one",
        )
        .copy()
    )

    target_month = (
        out[
            "target_period"
        ]
        .dt.month
    )

    out[
        "target_month_sin"
    ] = np.sin(
        2
        * np.pi
        * target_month
        / 12.0
    )

    out[
        "target_month_cos"
    ] = np.cos(
        2
        * np.pi
        * target_month
        / 12.0
    )

    for expert in ALL_EXPERTS:

        out[
            f"log_pred__{expert}"
        ] = np.log(
            out[expert]
            .clip(
                lower=1e-6
            )
        )

    out[
        "baseline_log_pred"
    ] = np.log(
        out[
            "baseline_pred"
        ]
        .clip(
            lower=1e-6
        )
    )

    baseline_log_cols = [
        f"log_pred__{e}"
        for e in BASELINE_EXPERTS
    ]

    all_log_cols = [
        f"log_pred__{e}"
        for e in ALL_EXPERTS
    ]

    out[
        "baseline_disagreement_std"
    ] = (
        out[
            baseline_log_cols
        ]
        .std(axis=1)
    )

    out[
        "baseline_disagreement_range"
    ] = (
        out[
            baseline_log_cols
        ]
        .max(axis=1)
        - out[
            baseline_log_cols
        ]
        .min(axis=1)
    )

    out[
        "all_disagreement_std"
    ] = (
        out[
            all_log_cols
        ]
        .std(axis=1)
    )

    out[
        "all_disagreement_range"
    ] = (
        out[
            all_log_cols
        ]
        .max(axis=1)
        - out[
            all_log_cols
        ]
        .min(axis=1)
    )

    out[
        "baseline_vs_level"
    ] = (
        out[
            "baseline_log_pred"
        ]
        - out[
            "level_log"
        ]
    )

    out[
        "panel_vs_baseline"
    ] = (
        out[
            "log_pred__panel_ar"
        ]
        - out[
            "baseline_log_pred"
        ]
    )

    out[
        "seasonal_vs_baseline"
    ] = (
        out[
            "log_pred__seasonal"
        ]
        - out[
            "baseline_log_pred"
        ]
    )

    out[
        "chronos_vs_baseline"
    ] = (
        out[
            "log_pred__chronos_relative"
        ]
        - out[
            "baseline_log_pred"
        ]
    )

    out[
        "ets_vs_baseline"
    ] = (
        out[
            "log_pred__ets_relative"
        ]
        - out[
            "baseline_log_pred"
        ]
    )

    out[
        "snaive_vs_baseline"
    ] = (
        out[
            "log_pred__seasonal_growth"
        ]
        - out[
            "baseline_log_pred"
        ]
    )

    rank_cols = [
        "level_log",
        "mom_log",
        "yoy_log",
        "trend3",
        "vol3",
        "vol6",
        "yoy_mean3",
        "baseline_log_pred",
        "baseline_vs_level",
        "baseline_disagreement_std",
        "baseline_disagreement_range",
        "all_disagreement_std",
        "all_disagreement_range",
        "panel_vs_baseline",
        "seasonal_vs_baseline",
        "chronos_vs_baseline",
        "ets_vs_baseline",
        "snaive_vs_baseline",
    ]

    for col in rank_cols:

        out[
            f"{col}_pct"
        ] = (
            out
            .groupby(
                [
                    "origin",
                    "h",
                ]
            )[col]
            .rank(
                method="average",
                pct=True,
            )
        )

    return out


cal_base = add_residual_features(
    calibration
)

bench_base = add_residual_features(
    benchmark
)

assert len(
    cal_base
) == 12_096

assert len(
    bench_base
) == 24_192

print(
    "Feature tables: PASSED"
)


## 6. Cross-sectional target

Target специально центрируется внутри каждой forecast-cell.

Это ограничивает модель: она учится только тому, какое МО относительно остальных должно быть чуть выше или ниже baseline.

In [ ]:
cal_base[
    "raw_log_residual"
] = np.log(
    cal_base[
        "actual_value"
    ]
    / cal_base[
        "baseline_pred"
    ]
)

cal_base[
    "cell_residual_median"
] = (
    cal_base
    .groupby(
        [
            "origin",
            "h",
        ]
    )[
        "raw_log_residual"
    ]
    .transform(
        "median"
    )
)

cal_base[
    "cs_residual_target"
] = (
    cal_base[
        "raw_log_residual"
    ]
    - cal_base[
        "cell_residual_median"
    ]
)

target_check = (
    cal_base
    .groupby(
        [
            "origin",
            "h",
        ]
    )[
        "cs_residual_target"
    ]
    .median()
)

assert np.allclose(
    target_check.to_numpy(),
    0.0,
    atol=1e-12,
)

print(
    "Cross-sectional target: PASSED"
)


## 7. CatBoost и pre-test OOF

Сетка намеренно небольшая.

Модель и сила поправки `alpha` выбираются только на May–June blocked OOF.

Alpha=0 разрешён. Если локальная поправка не помогает для горизонта, она автоматически отключается.

In [ ]:
NUMERIC_FEATURES = [
    "h",
    "level_log",
    "mom_log",
    "yoy_log",
    "trend3",
    "vol3",
    "vol6",
    "yoy_mean3",
    "panel_mom_median",
    "panel_yoy_median",
    "panel_mom_mad",
    "panel_yoy_mad",
    "target_month_sin",
    "target_month_cos",
    "baseline_log_pred",
    "baseline_vs_level",
    "baseline_disagreement_std",
    "baseline_disagreement_range",
    "all_disagreement_std",
    "all_disagreement_range",
    "panel_vs_baseline",
    "seasonal_vs_baseline",
    "chronos_vs_baseline",
    "ets_vs_baseline",
    "snaive_vs_baseline",
    "level_log_pct",
    "mom_log_pct",
    "yoy_log_pct",
    "trend3_pct",
    "vol3_pct",
    "vol6_pct",
    "yoy_mean3_pct",
    "baseline_log_pred_pct",
    "baseline_vs_level_pct",
    "baseline_disagreement_std_pct",
    "baseline_disagreement_range_pct",
    "all_disagreement_std_pct",
    "all_disagreement_range_pct",
    "panel_vs_baseline_pct",
    "seasonal_vs_baseline_pct",
    "chronos_vs_baseline_pct",
    "ets_vs_baseline_pct",
    "snaive_vs_baseline_pct",
    "size_mean_2023",
]

CAT_FEATURES = [
    "size_group",
]

MODEL_FEATURES = (
    NUMERIC_FEATURES
    + CAT_FEATURES
)

MODEL_CONFIGS = {
    "shallow": {
        "iterations":
            300,
        "depth":
            4,
        "learning_rate":
            0.04,
        "l2_leaf_reg":
            20.0,
    },

    "balanced": {
        "iterations":
            450,
        "depth":
            5,
        "learning_rate":
            0.03,
        "l2_leaf_reg":
            25.0,
    },

    "smooth": {
        "iterations":
            600,
        "depth":
            4,
        "learning_rate":
            0.02,
        "l2_leaf_reg":
            35.0,
    },
}

ALPHA_GRID = np.round(
    np.arange(
        0.0,
        1.01,
        0.10,
    ),
    2,
)

print(
    "Features:",
    len(
        MODEL_FEATURES
    ),
)

print(
    "Configs:",
    list(
        MODEL_CONFIGS
    ),
)


In [ ]:
def fit_residual_model(
    train_df,
    config,
):
    # Учитываем абсолютную MAE-метрику,
    # но ограничиваем влияние крупнейших МО.
    scale = (
        train_df[
            "baseline_pred"
        ]
        / train_df[
            "baseline_pred"
        ].median()
    )

    sample_weight = np.sqrt(
        scale.clip(
            lower=1e-6
        )
    )

    sample_weight = np.clip(
        sample_weight,
        0.5,
        3.0,
    )

    model = CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="MAE",
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
        **config,
    )

    model.fit(
        train_df[
            MODEL_FEATURES
        ],
        train_df[
            "cs_residual_target"
        ],
        cat_features=
            CAT_FEATURES,
        sample_weight=
            sample_weight,
    )

    return model


def center_predictions(
    df,
    pred_col,
):
    # Центрируем только по самим predictions.
    out = df.copy()

    median_col = (
        f"{pred_col}_median"
    )

    centered_col = (
        f"{pred_col}_centered"
    )

    out[
        median_col
    ] = (
        out
        .groupby(
            [
                "origin",
                "h",
            ]
        )[pred_col]
        .transform(
            "median"
        )
    )

    out[
        centered_col
    ] = (
        out[pred_col]
        - out[
            median_col
        ]
    )

    return out


def evaluate_alpha(
    df,
    alpha,
):
    pred = (
        df[
            "baseline_pred"
        ]
        * np.exp(
            float(alpha)
            * df[
                "raw_model_residual_centered"
            ]
        )
    )

    return float(
        np.mean(
            np.abs(
                df[
                    "actual_value"
                ]
                - pred
            )
        )
    )


In [ ]:
OOF_VAL_MONTHS = [
    pd.Timestamp(
        "2024-05-01"
    ),
    pd.Timestamp(
        "2024-06-01"
    ),
]

config_oof = {}

for (
    config_name,
    config
) in MODEL_CONFIGS.items():

    print(
        "\nCONFIG:",
        config_name,
    )

    fold_parts = []

    for val_month in (
        OOF_VAL_MONTHS
    ):

        train_df = (
            cal_base[
                cal_base[
                    "target_period"
                ]
                < val_month
            ]
            .copy()
        )

        val_df = (
            cal_base[
                cal_base[
                    "target_period"
                ]
                == val_month
            ]
            .copy()
        )

        if train_df.empty:
            continue

        model = (
            fit_residual_model(
                train_df,
                config,
            )
        )

        val_df[
            "raw_model_residual"
        ] = model.predict(
            val_df[
                MODEL_FEATURES
            ]
        )

        val_df = (
            center_predictions(
                val_df,
                "raw_model_residual",
            )
        )

        fold_parts.append(
            val_df[
                [
                    "series_id",
                    "origin",
                    "h",
                    "target_period",
                    "actual_value",
                    "baseline_pred",
                    "baseline_error",
                    "raw_model_residual",
                    "raw_model_residual_centered",
                ]
            ]
        )

        print(
            " validation:",
            val_month.date(),
            "rows:",
            len(
                val_df
            ),
        )

    config_oof[
        config_name
    ] = pd.concat(
        fold_parts,
        ignore_index=True,
    )


In [ ]:
search_rows = []
alpha_details = {}

for (
    config_name,
    oof
) in config_oof.items():

    # Сначала общий alpha.
    global_scores = []

    for alpha in ALPHA_GRID:

        global_scores.append({
            "alpha":
                float(alpha),
            "MAE":
                evaluate_alpha(
                    oof,
                    alpha,
                ),
        })

    global_scores = pd.DataFrame(
        global_scores
    )

    best_global = (
        global_scores
        .sort_values("MAE")
        .iloc[0]
    )

    alpha_global = float(
        best_global[
            "alpha"
        ]
    )

    # Затем horizon-specific alpha
    # с shrinkage к общему значению.
    alpha_by_h = {}
    detail_rows = []

    for h in [1, 2, 3]:

        part = (
            oof[
                oof[
                    "h"
                ]
                == h
            ]
            .copy()
        )

        if part.empty:

            alpha_by_h[h] = (
                alpha_global
            )

            continue

        h_scores = []

        for alpha in (
            ALPHA_GRID
        ):

            h_scores.append({
                "alpha":
                    float(
                        alpha
                    ),
                "MAE":
                    evaluate_alpha(
                        part,
                        alpha,
                    ),
            })

        h_scores = pd.DataFrame(
            h_scores
        )

        best_h = (
            h_scores
            .sort_values(
                "MAE"
            )
            .iloc[0]
        )

        raw_alpha_h = float(
            best_h[
                "alpha"
            ]
        )

        n_cells = (
            part[
                [
                    "origin",
                    "h",
                    "target_period",
                ]
            ]
            .drop_duplicates()
            .shape[0]
        )

        rho = (
            n_cells
            / (
                n_cells
                + 2.0
            )
        )

        final_alpha_h = (
            rho
            * raw_alpha_h
            + (
                1.0
                - rho
            )
            * alpha_global
        )

        alpha_by_h[h] = float(
            final_alpha_h
        )

        detail_rows.append({
            "h":
                h,
            "n_cells":
                n_cells,
            "raw_alpha_h":
                raw_alpha_h,
            "rho":
                rho,
            "final_alpha_h":
                final_alpha_h,
        })

    pred_parts = []

    for (
        h,
        part
    ) in oof.groupby(
        "h"
    ):

        alpha_h = (
            alpha_by_h[
                int(h)
            ]
        )

        temp = part.copy()

        temp[
            "corrected_pred"
        ] = (
            temp[
                "baseline_pred"
            ]
            * np.exp(
                alpha_h
                * temp[
                    "raw_model_residual_centered"
                ]
            )
        )

        temp[
            "corrected_error"
        ] = (
            temp[
                "actual_value"
            ]
            - temp[
                "corrected_pred"
            ]
        ).abs()

        pred_parts.append(
            temp
        )

    oof_final = pd.concat(
        pred_parts,
        ignore_index=True,
    )

    baseline_oof_mae = float(
        oof_final[
            "baseline_error"
        ].mean()
    )

    corrected_oof_mae = float(
        oof_final[
            "corrected_error"
        ].mean()
    )

    search_rows.append({
        "config":
            config_name,
        "alpha_global":
            alpha_global,
        "OOF_baseline_MAE":
            baseline_oof_mae,
        "OOF_corrected_MAE":
            corrected_oof_mae,
        "OOF_gain":
            baseline_oof_mae
            - corrected_oof_mae,
    })

    alpha_details[
        config_name
    ] = {
        "alpha_global":
            alpha_global,
        "alpha_by_h":
            alpha_by_h,
        "details":
            pd.DataFrame(
                detail_rows
            ),
        "oof":
            oof_final,
    }


oof_search = (
    pd.DataFrame(
        search_rows
    )
    .sort_values(
        "OOF_corrected_MAE"
    )
    .reset_index(
        drop=True
    )
)

display(
    oof_search.round(3)
)


In [ ]:
best = (
    oof_search.iloc[0]
)

BEST_CONFIG_NAME = str(
    best[
        "config"
    ]
)

BEST_CONFIG = (
    MODEL_CONFIGS[
        BEST_CONFIG_NAME
    ]
)

ALPHA_GLOBAL = float(
    alpha_details[
        BEST_CONFIG_NAME
    ][
        "alpha_global"
    ]
)

ALPHA_BY_H = (
    alpha_details[
        BEST_CONFIG_NAME
    ][
        "alpha_by_h"
    ]
)

print(
    "Frozen config:",
    BEST_CONFIG_NAME,
)

print(
    "Global alpha:",
    round(
        ALPHA_GLOBAL,
        3,
    ),
)

print(
    "Alpha by horizon:",
    {
        h:
            round(
                alpha,
                3,
            )
        for (
            h,
            alpha
        ) in ALPHA_BY_H.items()
    },
)

print(
    "Pre-test OOF gain:",
    round(
        float(
            best[
                "OOF_gain"
            ]
        ),
        3,
    ),
)

display(
    alpha_details[
        BEST_CONFIG_NAME
    ][
        "details"
    ]
    .round(3)
)

print(
    "Residual model frozen: PASSED"
)


## 8. Jun–Sep benchmark

После заморозки CatBoost обучается на историческом calibration-периоде (включая target_period = 2024-06-01).

Фактические значения июля–декабря не участвуют в fit или выборе alpha;
данные за июнь входят в ретроспективную калибровку. Это не real-time holdout на 1 июня.

In [ ]:
final_residual_model = (
    fit_residual_model(
        cal_base,
        BEST_CONFIG,
    )
)

bench_eval = (
    bench_base.copy()
)

bench_eval[
    "raw_model_residual"
] = (
    final_residual_model
    .predict(
        bench_eval[
            MODEL_FEATURES
        ]
    )
)

bench_eval = (
    center_predictions(
        bench_eval,
        "raw_model_residual",
    )
)

bench_parts = []

for (
    h,
    part
) in bench_eval.groupby(
    "h"
):

    h = int(h)

    alpha_h = (
        ALPHA_BY_H[h]
    )

    temp = part.copy()

    temp["alpha"] = (
        alpha_h
    )

    temp[
        "corrected_pred"
    ] = (
        temp[
            "baseline_pred"
        ]
        * np.exp(
            alpha_h
            * temp[
                "raw_model_residual_centered"
            ]
        )
    )

    temp[
        "corrected_error"
    ] = (
        temp[
            "actual_value"
        ]
        - temp[
            "corrected_pred"
        ]
    ).abs()

    bench_parts.append(
        temp
    )

residual_challenger = pd.concat(
    bench_parts,
    ignore_index=True,
)

baseline_mae = float(
    residual_challenger[
        "baseline_error"
    ].mean()
)

residual_mae = float(
    residual_challenger[
        "corrected_error"
    ].mean()
)

gain = (
    baseline_mae
    - residual_mae
)

gain_pct = (
    100.0
    * gain
    / baseline_mae
)

benchmark_summary = pd.DataFrame([
    {
        "model":
            "Baseline ensemble",
        "MAE":
            baseline_mae,
    },
    {
        "model":
            "Cross-sectional residual correction",
        "MAE":
            residual_mae,
    },
]).sort_values(
    "MAE"
)

display(
    benchmark_summary.round(3)
)

print(
    "Выигрыш:",
    round(
        gain,
        3,
    ),
    "MAE",
)

print(
    "Относительное улучшение:",
    f"{gain_pct:.2f}%",
)

assert np.isclose(
    baseline_mae,
    761.916,
    atol=0.25,
)

assert np.isclose(
    residual_mae,
    758.339,
    atol=0.35,
)

print(
    "Jun–Sep benchmark: PASSED"
)


In [ ]:
benchmark_by_horizon = (
    residual_challenger
    .groupby(
        "h"
    )
    .agg(
        baseline_MAE=(
            "baseline_error",
            "mean",
        ),
        residual_MAE=(
            "corrected_error",
            "mean",
        ),
    )
)

benchmark_by_horizon[
    "gain"
] = (
    benchmark_by_horizon[
        "baseline_MAE"
    ]
    - benchmark_by_horizon[
        "residual_MAE"
    ]
)

benchmark_by_origin = (
    residual_challenger
    .groupby(
        "origin"
    )
    .agg(
        baseline_MAE=(
            "baseline_error",
            "mean",
        ),
        residual_MAE=(
            "corrected_error",
            "mean",
        ),
    )
)

benchmark_by_origin[
    "gain"
] = (
    benchmark_by_origin[
        "baseline_MAE"
    ]
    - benchmark_by_origin[
        "residual_MAE"
    ]
)

print(
    "По горизонту:"
)

display(
    benchmark_by_horizon
    .round(3)
)

print(
    "По origin:"
)

display(
    benchmark_by_origin
    .round(3)
)


### Академический вывод по Jun–Sep

Residual correction снижает MAE с **761.916 до 758.339**.

Улучшение небольшое — около **0.47%**, поэтому его нельзя трактовать как радикальный скачок качества.

При этом важна структура результата:

- основной выигрыш возникает на `h1`;
- для `h2` и `h3` pre-test OOF фактически предпочитает не корректировать baseline;
- следовательно, shrinkage работает как защита от переобучения.

Это поддерживает более общий вывод: при короткой временной истории полезнее ограничивать ML-модель локальной residual-поправкой, чем разрешать ей полностью перестраивать сильный ансамбль.

## 9. Bootstrap для residual correction

Пересэмплируем целые муниципальные ряды, сохраняя все их forecast-наблюдения вместе.

Так проверяется, не получен ли выигрыш за счёт нескольких отдельных МО.

In [ ]:
series_eval = (
    residual_challenger
    .groupby(
        "series_id"
    )
    .agg(
        baseline_error=(
            "baseline_error",
            "mean",
        ),
        residual_error=(
            "corrected_error",
            "mean",
        ),
    )
)

series_eval[
    "gain"
] = (
    series_eval[
        "baseline_error"
    ]
    - series_eval[
        "residual_error"
    ]
)

values = (
    series_eval[
        "gain"
    ]
    .to_numpy(
        dtype=float
    )
)

rng = np.random.default_rng(
    42
)

N_BOOT = 5000
n = len(
    values
)

boot_values = np.empty(
    N_BOOT
)

for b in range(
    N_BOOT
):

    idx = rng.integers(
        0,
        n,
        size=n,
    )

    boot_values[b] = (
        values[
            idx
        ]
        .mean()
    )

lo, hi = np.quantile(
    boot_values,
    [
        0.025,
        0.975,
    ],
)

residual_bootstrap = pd.DataFrame([{
    "mean_gain":
        values.mean(),
    "ci_2.5":
        lo,
    "ci_97.5":
        hi,
    "P(gain>0)":
        (
            boot_values
            > 0
        ).mean(),
    "share_series_improved":
        (
            values
            > 0
        ).mean(),
}])

display(
    residual_bootstrap
    .round(3)
)


## 10. Диагностика residual-сигнала

Проверяем, есть ли связь между прогнозируемой локальной поправкой и фактическим cross-sectional residual baseline.

Эта диагностика выполняется после benchmark и не используется для настройки модели.

In [ ]:
residual_challenger[
    "actual_log_residual"
] = np.log(
    residual_challenger[
        "actual_value"
    ]
    / residual_challenger[
        "baseline_pred"
    ]
)

residual_challenger[
    "actual_cs_residual"
] = (
    residual_challenger[
        "actual_log_residual"
    ]
    - residual_challenger
    .groupby(
        [
            "origin",
            "h",
        ]
    )[
        "actual_log_residual"
    ]
    .transform(
        "median"
    )
)

corr_rows = []

for (
    origin,
    h
), part in (
    residual_challenger
    .groupby(
        [
            "origin",
            "h",
        ]
    )
):

    corr_rows.append({
        "origin":
            origin,
        "h":
            h,
        "pearson":
            part[
                "raw_model_residual_centered"
            ]
            .corr(
                part[
                    "actual_cs_residual"
                ],
                method="pearson",
            ),
        "spearman":
            part[
                "raw_model_residual_centered"
            ]
            .corr(
                part[
                    "actual_cs_residual"
                ],
                method="spearman",
            ),
    })

correlations = pd.DataFrame(
    corr_rows
)

display(
    correlations.round(3)
)

print(
    "Mean Pearson:",
    round(
        correlations[
            "pearson"
        ].mean(),
        3,
    ),
)

print(
    "Mean Spearman:",
    round(
        correlations[
            "spearman"
        ].mean(),
        3,
    ),
)

feature_importance = (
    pd.DataFrame({
        "feature":
            MODEL_FEATURES,
        "importance":
            final_residual_model
            .get_feature_importance(),
    })
    .sort_values(
        "importance",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)

print(
    "\nTop-20 признаков:"
)

display(
    feature_importance
    .head(20)
    .round(3)
)


### Вывод по диагностике

Средняя корреляция невысокая, то есть остаточный сигнал слабый.

Это соответствует небольшому размеру итогового улучшения и объясняет, почему агрессивные ML-поправки в дополнительных экспериментах были нестабильны.

Наиболее важными оказываются признаки недавней YoY-динамики, положения МО относительно панели и расхождения baseline с альтернативными экспертами.

# Часть B. Online adaptive ensemble

## 11. Идея

Residual correction улучшает стандартный benchmark, но есть другая практическая задача: модель должна переноситься на следующий временной режим.

Для этого используем пять экспертов и на каждом новом origin подбираем неотрицательные веса с суммой 1.

При расчёте весов разрешены только уже завершившиеся прогнозы того же горизонта:

`target_period <= current origin`

Таким образом веса могут меняться со временем без использования будущего.

In [ ]:
def fit_mae_simplex(
    train_df,
    experts=EXPERTS,
):
    # Неотрицательные веса с суммой 1.
    X = train_df[
        experts
    ].to_numpy(
        dtype=float
    )

    y = train_df[
        "actual_value"
    ].to_numpy(
        dtype=float
    )

    def objective(w):

        return np.mean(
            np.abs(
                y
                - X @ w
            )
        )

    starts = [
        np.ones(
            len(experts)
        )
        / len(experts),

        np.array([
            0.45,
            0.55,
            0.0,
            0.0,
            0.0,
        ]),

        np.array([
            0.0,
            0.0,
            1 / 3,
            1 / 3,
            1 / 3,
        ]),
    ]

    bounds = [
        (0.0, 1.0)
        for _ in experts
    ]

    constraints = [{
        "type":
            "eq",
        "fun":
            lambda w:
                np.sum(w)
                - 1.0,
    }]

    solutions = []

    for start in starts:

        result = minimize(
            objective,
            x0=start,
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
            options={
                "maxiter":
                    1000,
                "ftol":
                    1e-9,
                "disp":
                    False,
            },
        )

        if result.success:

            w = np.clip(
                result.x,
                0.0,
                1.0,
            )

            w = (
                w
                / w.sum()
            )

            solutions.append(
                (
                    objective(w),
                    w,
                )
            )

    if not solutions:

        raise RuntimeError(
            "Не удалось подобрать online-веса."
        )

    loss, weights = min(
        solutions,
        key=lambda x:
            x[0],
    )

    return (
        weights,
        loss,
    )


def online_forecast_cell(
    origin,
    h,
):
    # Одна forward-cell: origin x horizon.
    origin = pd.Timestamp(
        origin
    )

    h = int(h)

    current = (
        expert_history[
            (
                expert_history[
                    "origin"
                ]
                == origin
            )
            &
            (
                expert_history[
                    "h"
                ]
                == h
            )
        ]
        .copy()
    )

    history = (
        expert_history[
            (
                expert_history[
                    "h"
                ]
                == h
            )
            &
            (
                expert_history[
                    "target_period"
                ]
                <= origin
            )
        ]
        .copy()
    )

    resolved_cells = (
        history[
            "target_period"
        ]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    if not resolved_cells:

        raise ValueError(
            "Нет завершённой истории для online-ансамбля."
        )

    weights, train_mae = (
        fit_mae_simplex(
            history
        )
    )

    prediction = (
        current[
            EXPERTS
        ]
        .to_numpy(
            dtype=float
        )
        @ weights
    )

    out = (
        current[
            [
                "series_id",
                "origin",
                "h",
                "target_period",
                "actual_value",
            ]
        ]
        .copy()
    )

    out[
        "online_pred"
    ] = prediction

    out[
        "online_error"
    ] = (
        out[
            "actual_value"
        ]
        - out[
            "online_pred"
        ]
    ).abs()

    weight_row = {
        "origin":
            origin,
        "h":
            h,
        "n_history_cells":
            len(
                resolved_cells
            ),
        "train_mae":
            train_mae,
    }

    for (
        expert,
        weight
    ) in zip(
        EXPERTS,
        weights,
    ):

        weight_row[
            f"w_{expert}"
        ] = weight

    return (
        out,
        weight_row,
    )


## 12. Поздняя forward-проверка

Используем три forecast-cell:

- Oct origin, h1 -> Nov;
- Oct origin, h2 -> Dec;
- Nov origin, h1 -> Dec.

Всего: 6 048 прогнозов.

In [ ]:
forward_cells = [
    (
        pd.Timestamp(
            "2024-10-01"
        ),
        1,
    ),
    (
        pd.Timestamp(
            "2024-10-01"
        ),
        2,
    ),
    (
        pd.Timestamp(
            "2024-11-01"
        ),
        1,
    ),
]

online_parts = []
weight_rows = []

for origin, h in (
    forward_cells
):

    pred, weights = (
        online_forecast_cell(
            origin,
            h,
        )
    )

    online_parts.append(
        pred
    )

    weight_rows.append(
        weights
    )

online_forward = pd.concat(
    online_parts,
    ignore_index=True,
)

online_weights = pd.DataFrame(
    weight_rows
)

display(
    online_weights.round(3)
)

assert len(
    online_forward
) == 6_048


In [ ]:
# Static 45/55 из notebook 02.
forward_static = (
    static_forward[
        [
            "series_id",
            "origin",
            "horizon",
            "target_period",
            "actual_value",
            "final_pred",
        ]
    ]
    .rename(
        columns={
            "horizon":
                "h",
            "final_pred":
                "static_pred",
        }
    )
)

# Сильный baseline ensemble на этом же forward-периоде.
forward_baseline = (
    forward_history[
        [
            "series_id",
            "origin",
            "h",
            "target_period",
            "chronos_relative",
            "ets_relative",
            "seasonal_growth",
        ]
    ]
    .copy()
)

forward_baseline[
    "baseline_pred"
] = (
    forward_baseline[
        [
            "chronos_relative",
            "ets_relative",
            "seasonal_growth",
        ]
    ]
    .mean(axis=1)
)

forward_compare = (
    online_forward
    .merge(
        forward_static,
        on=[
            "series_id",
            "origin",
            "h",
            "target_period",
            "actual_value",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        forward_baseline[
            [
                "series_id",
                "origin",
                "h",
                "target_period",
                "baseline_pred",
            ]
        ],
        on=[
            "series_id",
            "origin",
            "h",
            "target_period",
        ],
        how="left",
        validate="one_to_one",
    )
)

forward_compare[
    "static_error"
] = (
    forward_compare[
        "actual_value"
    ]
    - forward_compare[
        "static_pred"
    ]
).abs()

forward_compare[
    "baseline_error"
] = (
    forward_compare[
        "actual_value"
    ]
    - forward_compare[
        "baseline_pred"
    ]
).abs()

forward_summary = pd.Series({
    "Online adaptive":
        forward_compare[
            "online_error"
        ].mean(),

    "Static 45/55":
        forward_compare[
            "static_error"
        ].mean(),

    "Baseline ensemble":
        forward_compare[
            "baseline_error"
        ].mean(),
}).sort_values()

display(
    forward_summary
    .round(3)
    .to_frame(
        "MAE"
    )
)

assert np.isclose(
    forward_summary[
        "Online adaptive"
    ],
    834.497,
    atol=0.25,
)

assert np.isclose(
    forward_summary[
        "Static 45/55"
    ],
    858.810,
    atol=0.25,
)

assert np.isclose(
    forward_summary[
        "Baseline ensemble"
    ],
    880.397,
    atol=0.30,
)

print(
    "Forward benchmark: PASSED"
)


In [ ]:
forward_by_cell = (
    forward_compare
    .groupby(
        [
            "origin",
            "h",
        ]
    )
    .agg(
        online_MAE=(
            "online_error",
            "mean",
        ),
        static_MAE=(
            "static_error",
            "mean",
        ),
        baseline_MAE=(
            "baseline_error",
            "mean",
        ),
    )
)

display(
    forward_by_cell
    .round(3)
)


### Академический вывод по forward-периоду

На более позднем периоде:

- baseline ensemble: **880.397 MAE**
- static 45/55: **858.810 MAE**
- online adaptive: **834.497 MAE**

Online adaptive снижает MAE относительно baseline примерно на **45.9**, то есть примерно на **5.2%**.

Это более крупный эффект, чем Jun–Sep residual correction.

Смысл результата не в том, что `834.497` лучше `758.339`: эти значения относятся к разным временным окнам и напрямую не сравниваются.

Корректный вывод другой: при смене временного режима адаптивное перераспределение весов пяти экспертов оказалось устойчивее фиксированных смесей.

## 13. Bootstrap для online adaptive

Проверяем устойчивость выигрыша по муниципальным рядам.

In [ ]:
forward_series_eval = (
    forward_compare[
        [
            "series_id",
            "online_error",
            "static_error",
            "baseline_error",
        ]
    ]
    .groupby(
        "series_id"
    )
    .mean()
)

forward_series_eval[
    "gain_vs_static"
] = (
    forward_series_eval[
        "static_error"
    ]
    - forward_series_eval[
        "online_error"
    ]
)

forward_series_eval[
    "gain_vs_baseline"
] = (
    forward_series_eval[
        "baseline_error"
    ]
    - forward_series_eval[
        "online_error"
    ]
)

rng = np.random.default_rng(
    42
)

N_BOOT = 5000
n = len(
    forward_series_eval
)


def bootstrap_gain(
    values,
):
    values = np.asarray(
        values,
        dtype=float,
    )

    sims = np.empty(
        N_BOOT
    )

    for b in range(
        N_BOOT
    ):

        idx = rng.integers(
            0,
            n,
            size=n,
        )

        sims[b] = (
            values[
                idx
            ]
            .mean()
        )

    lo, hi = np.quantile(
        sims,
        [
            0.025,
            0.975,
        ],
    )

    return {
        "mean_gain":
            values.mean(),
        "ci_2.5":
            lo,
        "ci_97.5":
            hi,
        "P(gain>0)":
            (
                sims > 0
            ).mean(),
    }


forward_bootstrap = (
    pd.DataFrame({
        "vs_static":
            bootstrap_gain(
                forward_series_eval[
                    "gain_vs_static"
                ]
            ),

        "vs_baseline":
            bootstrap_gain(
                forward_series_eval[
                    "gain_vs_baseline"
                ]
            ),
    })
    .T
)

display(
    forward_bootstrap.round(3)
)

assert (
    forward_bootstrap.loc[
        "vs_static",
        "ci_2.5",
    ]
    > 0
)

assert (
    forward_bootstrap.loc[
        "vs_baseline",
        "ci_2.5",
    ]
    > 0
)


## 14. Итоговая интерпретация forecasting-части

Примечание по воспроизводимости: модель, параметры и расчёт метрик не изменены
при cleanup. Перед обновлением `06` необходимо сверить forward MAE
с пересчитанным `forecast_final_summary.csv` и manifest; в текущем
ноутбуке сохранено значение **834.497**.

Получены два взаимодополняющих результата.

### 1. Стандартный Jun–Sep benchmark

Cross-sectional residual correction немного улучшает сильный baseline:

`761.916 -> 758.339 MAE`

Эффект небольшой, поэтому его следует трактовать как аккуратное улучшение, а не как принципиально новый уровень точности.

### 2. Более поздний forward-период

Online adaptive ensemble показывает более заметный выигрыш:

`880.397 -> 834.331 MAE`

Здесь преимущество подтверждается bootstrap по муниципальным рядам.

### Общий вывод

На короткой временной истории наиболее устойчивыми оказались две стратегии:

- сильный baseline нужно корректировать консервативно, только по локальному residual;
- при смене временного режима полезно адаптивно пересчитывать глобальные веса экспертов по уже наблюдаемым ошибкам.

Более агрессивные ML-routing и дополнительные усложнения в финальную модель не включаются, потому что не показали устойчивого дополнительного выигрыша.

## 15. Сохранение финальных forecasting-артефактов

In [ ]:
EXPERT_HISTORY_FILE = (
    INTERIM_DIR
    / "adaptive_expert_history.parquet"
)

RESIDUAL_PRED_FILE = (
    INTERIM_DIR
    / "forecast_residual_challenger_predictions.parquet"
)

ONLINE_PRED_FILE = (
    INTERIM_DIR
    / "forecast_online_forward_predictions.parquet"
)

RESIDUAL_BOOT_FILE = (
    INTERIM_DIR
    / "forecast_residual_bootstrap.csv"
)

ONLINE_BOOT_FILE = (
    INTERIM_DIR
    / "forecast_online_bootstrap.csv"
)

IMPORTANCE_FILE = (
    INTERIM_DIR
    / "forecast_residual_feature_importance.csv"
)

SUMMARY_FILE = (
    INTERIM_DIR
    / "forecast_final_summary.csv"
)

MANIFEST_FILE = (
    INTERIM_DIR
    / "forecast_final_manifest.json"
)

expert_history.to_parquet(
    EXPERT_HISTORY_FILE,
    index=False,
)

residual_challenger.to_parquet(
    RESIDUAL_PRED_FILE,
    index=False,
)

forward_compare.to_parquet(
    ONLINE_PRED_FILE,
    index=False,
)

residual_bootstrap.to_csv(
    RESIDUAL_BOOT_FILE,
    index=False,
)

forward_bootstrap.to_csv(
    ONLINE_BOOT_FILE,
    index=True,
)

feature_importance.to_csv(
    IMPORTANCE_FILE,
    index=False,
)

final_summary = pd.DataFrame([
    {
        "evaluation":
            "Jun-Sep benchmark",
        "model":
            "Baseline ensemble",
        "MAE":
            baseline_mae,
    },
    {
        "evaluation":
            "Jun-Sep benchmark",
        "model":
            "Cross-sectional residual correction",
        "MAE":
            residual_mae,
    },
    {
        "evaluation":
            "Oct-Nov forward",
        "model":
            "Baseline ensemble",
        "MAE":
            float(
                forward_summary[
                    "Baseline ensemble"
                ]
            ),
    },
    {
        "evaluation":
            "Oct-Nov forward",
        "model":
            "Static 45/55",
        "MAE":
            float(
                forward_summary[
                    "Static 45/55"
                ]
            ),
    },
    {
        "evaluation":
            "Oct-Nov forward",
        "model":
            "Online adaptive",
        "MAE":
            float(
                forward_summary[
                    "Online adaptive"
                ]
            ),
    },
])

final_summary.to_csv(
    SUMMARY_FILE,
    index=False,
)

manifest = {
    "name":
        "Final forecasting pipeline",

    "n_series":
        2016,

    "jun_sep": {
        "baseline_mae":
            baseline_mae,
        "residual_correction_mae":
            residual_mae,
        "gain":
            baseline_mae
            - residual_mae,
        "interpretation":
            "challenger benchmark; not a new untouched holdout",
    },

    "oct_nov_forward": {
        "baseline_mae":
            float(
                forward_summary[
                    "Baseline ensemble"
                ]
            ),
        "static_mae":
            float(
                forward_summary[
                    "Static 45/55"
                ]
            ),
        "online_adaptive_mae":
            float(
                forward_summary[
                    "Online adaptive"
                ]
            ),
        "interpretation":
            "separate later forward evaluation",
    },

    "residual_model": {
        "config":
            BEST_CONFIG_NAME,
        "alpha_global":
            ALPHA_GLOBAL,
        "alpha_by_h":
            {
                str(h):
                    float(alpha)
                for (
                    h,
                    alpha
                ) in ALPHA_BY_H.items()
            },
    },

    "methodology":
        (
            "Forecast inputs are restricted to each origin. "
            "Retrospective residual calibration includes June 2024 targets; "
            "Jun-Sep results are challenger benchmark, not real-time June holdout. "
            "Online adaptive weights use only already resolved same-horizon forecasts."
        ),
}

with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        manifest,
        f,
        ensure_ascii=False,
        indent=2,
    )

artifacts = pd.DataFrame([
    {
        "artifact":
            EXPERT_HISTORY_FILE.name,
        "exists":
            EXPERT_HISTORY_FILE.exists(),
    },
    {
        "artifact":
            RESIDUAL_PRED_FILE.name,
        "exists":
            RESIDUAL_PRED_FILE.exists(),
    },
    {
        "artifact":
            ONLINE_PRED_FILE.name,
        "exists":
            ONLINE_PRED_FILE.exists(),
    },
    {
        "artifact":
            RESIDUAL_BOOT_FILE.name,
        "exists":
            RESIDUAL_BOOT_FILE.exists(),
    },
    {
        "artifact":
            ONLINE_BOOT_FILE.name,
        "exists":
            ONLINE_BOOT_FILE.exists(),
    },
    {
        "artifact":
            IMPORTANCE_FILE.name,
        "exists":
            IMPORTANCE_FILE.exists(),
    },
    {
        "artifact":
            SUMMARY_FILE.name,
        "exists":
            SUMMARY_FILE.exists(),
    },
    {
        "artifact":
            MANIFEST_FILE.name,
        "exists":
            MANIFEST_FILE.exists(),
    },
])

display(
    artifacts
)

assert artifacts[
    "exists"
].all()

print(
    "Финальные forecasting-артефакты сохранены в:",
    INTERIM_DIR,
)


# Финальный вывод

Forecasting-блок завершён.

Основной результат на стандартном benchmark:

**MAE 758.339 против baseline 761.916.**

Дополнительная поздняя forward-проверка показывает преимущество online adaptive ensemble:

**MAE 834.331 против baseline 880.397.**

Дальнейшее усложнение forecasting-части не требуется. Следующий этап проекта — обнаружение структурных шоков.